# Chapter 11: Large Language Models & Transformers
## Notebook 03 — Advanced LLMs

This notebook covers what happens **after** pretraining. We study **decoding strategies** (greedy, temperature, top-k, top-p), peek at the **KV cache**, develop intuition for **scaling laws**, walk through **evaluation** (perplexity, BLEU/ROUGE, win-rate, LLM-as-judge), and sketch the architecture of an **LLM-powered application**.

### What you'll learn

| Topic | Section |
|-------|--------|
| Decoding: greedy, temperature, top-k, top-p | §2 |
| Repetition penalty and stopping | §3 |
| KV cache concept and shapes | §4 |
| Scaling laws intuition | §5 |
| Evaluation: perplexity, BLEU/ROUGE, win-rate, LLM-as-judge | §6 |
| Building LLM apps: chunking, streaming, function calling | §7 |
| Capstone design and hand-off to Ch 12 / 13 / 14 | §8 |

**Estimated time:** 2.5 hours

---
*Generated by Berta AI*

---
## 1. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

print("Setup complete.")

---
## 2. Decoding Strategies

Generation is **iterative**: at each step the model produces logits over the vocabulary, we pick a token, append it, and repeat.

| Strategy | What it does | When to use |
|----------|--------------|-------------|
| **Greedy** | Always argmax | Deterministic, can loop |
| **Temperature** | Scale logits by `1/T` before softmax | Tune confidence: `T<1` sharpens, `T>1` flattens |
| **Top-k** | Sample only from the top `k` tokens | Avoids long tail of garbage |
| **Top-p (nucleus)** | Sample from the smallest set with cum. prob. ≥ `p` | Adaptive, usually best general default |

We'll exercise all of them on a toy logit vector.

In [ ]:
from generation_utils import (
    apply_temperature, sample_with_temperature,
    top_k_sample, top_p_sample, greedy_step,
)

vocab = ['cat', 'dog', 'bird', 'tree', 'car', 'sky', 'love', 'code']
logits = np.array([3.0, 2.5, 1.8, 1.5, 0.8, 0.4, -0.5, -1.0])

probs = np.exp(logits - logits.max()); probs /= probs.sum()
print("Token       prob")
for t, p in zip(vocab, probs):
    print(f"  {t:6}     {p:.3f}")
print("\nGreedy pick:", vocab[greedy_step(logits)])

In [ ]:
# Compare distributions under different strategies.
def empirical_dist(sampler, n=2000):
    rng = np.random.default_rng(0)
    counts = np.zeros(len(vocab))
    for _ in range(n):
        counts[sampler(rng)] += 1
    return counts / counts.sum()

dists = {
    'T = 0.5': empirical_dist(lambda r: sample_with_temperature(logits, 0.5, rng=r)),
    'T = 1.0': empirical_dist(lambda r: sample_with_temperature(logits, 1.0, rng=r)),
    'T = 2.0': empirical_dist(lambda r: sample_with_temperature(logits, 2.0, rng=r)),
    'top-k=3': empirical_dist(lambda r: top_k_sample(logits, k=3, rng=r)),
    'top-p=0.9': empirical_dist(lambda r: top_p_sample(logits, p=0.9, rng=r)),
}

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.16
for i, (name, d) in enumerate(dists.items()):
    ax.bar(np.arange(len(vocab)) + i * width, d, width=width, label=name)
ax.set_xticks(np.arange(len(vocab)) + 2 * width)
ax.set_xticklabels(vocab); ax.set_ylabel('empirical prob')
ax.set_title('Decoding strategies on the same logits')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## 3. Repetition Penalty & Stopping

Sampled outputs often **loop**. Two common fixes:

- **Repetition penalty** (Keskar et al.): divide the logit of any already-generated token by `penalty > 1`.
- **EOS stopping**: stop generation as soon as the end-of-sequence token is produced.
- **Max length cap** is mandatory in production.

In [ ]:
from generation_utils import apply_repetition_penalty, greedy_decode

# Toy autoregressive "model": logits depend slightly on the last token.
def toy_logits(seq):
    base = np.array([2.0, 1.0, 1.5, 0.5, 1.2, 0.8])  # vocab size 6
    last = seq[-1] if seq else 0
    base = base.copy()
    base[last] += 1.5  # likes to repeat
    return base

print("Without penalty:", greedy_decode(toy_logits, [0], max_new_tokens=8))

# Apply repetition penalty manually each step.
seq = [0]
for _ in range(8):
    lg = apply_repetition_penalty(toy_logits(seq), seq, penalty=1.5)
    seq.append(int(np.argmax(lg)))
print("With penalty:    ", seq)

---
## 4. The KV Cache

At step *t*, a decoder needs the keys and values of all *previous* tokens to compute attention. Recomputing them each step is **O(t²)** per step. The **KV cache** stores them so each step is O(t).

Shapes per layer:

```
K: (batch, num_heads, seq_so_far, d_head)
V: (batch, num_heads, seq_so_far, d_head)
```

Memory grows **linearly** with sequence length — and with the number of users / requests.

In [ ]:
# Illustrate KV cache shapes for typical configs.
def kv_cache_size(batch, heads, seq, d_head, n_layers, dtype_bytes=2):
    # K and V each
    bytes_total = 2 * batch * heads * seq * d_head * n_layers * dtype_bytes
    return bytes_total / 1e9  # GB

for cfg in [
    ("DistilBERT-ish", 1, 12, 512, 64, 6),
    ("GPT-2 small",   1, 12, 1024, 64, 12),
    ("Llama-7B-ish",  1, 32, 2048, 128, 32),
    ("Llama-7B 8-batch ctx 4k", 8, 32, 4096, 128, 32),
]:
    name, b, h, s, d, n = cfg
    print(f"{name:32s} -> {kv_cache_size(b, h, s, d, n):6.2f} GB (fp16)")

---
## 5. Scaling Laws

Loss decreases **as a power law** in compute, data, and parameters (Kaplan et al. 2020; Hoffmann et al. 2022, "Chinchilla"). Two practical takeaways:

1. **Compute-optimal models** balance parameters and tokens — Chinchilla showed earlier models were under-trained.
2. **Diminishing returns**: doubling compute reduces loss by a *constant* amount, not a constant fraction.

In [ ]:
# Toy loss curve: L(C) = a * C^-alpha + b
C = np.logspace(0, 6, 80)
loss = 4.0 * C ** -0.07 + 1.7

plt.figure(figsize=(6, 3.5))
plt.loglog(C, loss)
plt.xlabel('compute (arbitrary units, log scale)')
plt.ylabel('loss (log scale)')
plt.title('Power-law scaling: bigger compute -> lower loss')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6. Evaluating LLMs

| Metric | What it measures | Notes |
|--------|------------------|-------|
| **Perplexity** | `exp(-mean log p(token))` on held-out text | Lower = better; only meaningful within the same tokenizer |
| **BLEU / ROUGE** | n-gram overlap with a reference | Standard in MT / summarisation; rewards surface form |
| **Win-rate (A/B)** | Humans (or a strong model) pick A vs B | Most directly measures "is it better?" |
| **LLM-as-judge** | A strong LLM scores outputs against a rubric | Cheap and fast; biased toward verbose / similar-style answers |
| **Task-specific** | Exact match, F1, code-pass-rate, etc. | Use whenever the task is gradeable |

In [ ]:
from generation_utils import perplexity

# Simulate "model log-probabilities" for a held-out sentence.
log_probs = [-0.5, -1.2, -0.3, -2.0, -0.8]
print("Perplexity:", round(perplexity(log_probs), 3))

# Compare two models: lower PPL = better fit.
print("Model A PPL:", round(perplexity([-0.4, -0.7, -0.5, -0.6]), 3))
print("Model B PPL:", round(perplexity([-1.1, -1.4, -0.9, -1.2]), 3))

In [ ]:
# Win-rate sketch.
np.random.seed(0)
n = 100
wins_A = np.random.binomial(1, 0.62, size=n)  # 62% A-wins
print(f"Model A win-rate over {n} comparisons: {wins_A.mean():.2%}")
print("With n=100, 95% CI is roughly +/- 10pp — small wins need many comparisons.")

---
## 7. Building LLM-Powered Apps

The shape of nearly every LLM app is the same:

```
user_input
   |
   v
[ retrieve context ] -> [ compose prompt ] -> [ call LLM (streaming) ]
                                                        |
                                                        v
                                                 [ parse / tools ]
                                                        |
                                                        v
                                                  user_output
```

Three patterns to internalise:

1. **Chunking** — split long documents on token boundaries with overlap so retrieval / summarisation stay coherent.
2. **Streaming** — yield tokens as they arrive; first-token latency is the user-perceived latency.
3. **Function / tool calling** — let the LLM emit a structured JSON call that your app executes (calculator, search, DB).

In [ ]:
# Token-aware chunker (uses our fallback tokenizer; works without transformers).
from llm_utils import LLMTokenizerWrapper

tok = LLMTokenizerWrapper()

def chunk(text, chunk_size=20, overlap=5):
    ids = tok.encode(text)
    out = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(ids), step):
        out.append(ids[i:i + chunk_size])
        if i + chunk_size >= len(ids):
            break
    return out

doc = ("Transformers replaced RNNs in NLP. They use self-attention to relate "
       "every token to every other token in parallel. The Transformer was "
       "introduced in 2017 and is the foundation of modern LLMs.")
chunks = chunk(doc, chunk_size=15, overlap=4)
print(f"{len(chunks)} chunks of <=15 tokens, with 4-token overlap")
for i, c in enumerate(chunks):
    print(f"  chunk {i}: {len(c)} tokens, ids[:8]={c[:8]}")

In [ ]:
# Streaming sketch: a generator that yields one decoded token at a time.
def stream_generate(prompt_ids, n=8, vocab=('the','cat','sat','on','mat','quietly','today','.')):
    rng = np.random.default_rng(0)
    seq = list(prompt_ids)
    for _ in range(n):
        # pretend the model returns logits each step
        logits = rng.standard_normal(len(vocab))
        nxt = int(np.argmax(logits))
        seq.append(nxt)
        yield vocab[nxt]

print("Streaming output:", end=' ')
for tok_str in stream_generate([0]):
    print(tok_str, end=' ', flush=True)
print()

---
## 8. Capstone Design & What's Next

A nice end-to-end project that uses everything in this chapter:

1. **Pick a corpus** (`datasets/sample_corpus.txt` or your own).
2. **Embed** every paragraph with `EmbeddingExtractor`.
3. **Search**: given a user query, return top-k most similar paragraphs (`top_k_similar`).
4. **Compose** a prompt: `"Answer using only these passages: {top_k}. Question: {query}"`.
5. **Generate**: send to an LLM (or a local one). Stream the response. Apply a stop sequence.
6. **Evaluate**: a small win-rate study comparing top-3 vs top-5 retrieval.

You have just sketched a **RAG (Retrieval-Augmented Generation)** system — which is the topic of **Chapter 13**.

### Hand-off

- **Chapter 12 — Prompt Engineering**: how to *steer* the LLM you now know how to *call*.
- **Chapter 13 — Retrieval-Augmented Generation**: scale §7's "chunk + embed + search + answer" pipeline to real corpora.
- **Chapter 14 — Fine-Tuning**: when prompting and retrieval are not enough, change the model itself.

---
## 9. Key Takeaways

- Decoding choice (`temperature`, `top-k`, `top-p`, repetition penalty) shapes the output as much as the model.
- The **KV cache** is the workhorse that makes long-context decoding tractable, and the dominant memory cost.
- Scaling laws are predictable: more compute → lower loss, but with diminishing returns.
- Evaluate with the right tool: perplexity for LM fit, BLEU/ROUGE for surface match, **win-rate** for "is it better?", and remember LLM-as-judge has its own biases.
- LLM apps share a common shape: chunk, embed, retrieve, compose, generate (stream), tools.

---
*Generated by Berta AI*